In [6]:
import pandas as pd
import numpy as np
import pyarrow

print("Python environment is ready!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("PyArrow version:", pyarrow.__version__)

Python environment is ready!
Pandas version: 3.1.1
NumPy version: 2.4.6
PyArrow version: 26.0.1


In [7]:

# This notebook explores the public Multi-Benchmark LLM Agent Traces dataset.

# Objectives

# - Understand the dataset structure.
# - Identify available files and fields.
# - Explore traces and nested spans.
# - Identify information useful for agent monitoring.
# - Prepare the dataset for the Bronze, Silver, and Gold layers.

In [8]:
from huggingface_hub import HfApi
DATASET_NAME = "Exgentic/agent-llm-traces"
api = HfApi()
dataset_info = api.dataset_info(DATASET_NAME)
print("Dataset:", DATASET_NAME)
print("\nAvailable files:")
for file in dataset_info.siblings:
    print("-", file.rfilename)

Dataset: Exgentic/agent-llm-traces

Available files:
- .gitattributes
- README.md
- data/train-00000-of-00039.parquet
- data/train-00001-of-00039.parquet
- data/train-00002-of-00039.parquet
- data/train-00003-of-00039.parquet
- data/train-00004-of-00039.parquet
- data/train-00005-of-00039.parquet
- data/train-00006-of-00039.parquet
- data/train-00007-of-00039.parquet
- data/train-00008-of-00039.parquet
- data/train-00009-of-00039.parquet
- data/train-00010-of-00039.parquet
- data/train-00011-of-00039.parquet
- data/train-00012-of-00039.parquet
- data/train-00013-of-00039.parquet
- data/train-00014-of-00039.parquet
- data/train-00015-of-00039.parquet
- data/train-00016-of-00039.parquet
- data/train-00017-of-00039.parquet
- data/train-00018-of-00039.parquet
- data/train-00019-of-00039.parquet
- data/train-00020-of-00039.parquet
- data/train-00021-of-00039.parquet
- data/train-00022-of-00039.parquet
- data/train-00023-of-00039.parquet
- data/train-00024-of-00039.parquet
- data/train-00025

In [9]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

DATASET_NAME = "Exgentic/agent-llm-traces"

# Download only the first Parquet file
file_path = hf_hub_download(
    repo_id=DATASET_NAME,
    filename="data/train-00000-of-00039.parquet",
    repo_type="dataset"
)

print("Downloaded file:", file_path)

# Read Parquet metadata
parquet_file = pq.ParquetFile(file_path)

print("\nNumber of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)

print("\nDataset schema:")
print(parquet_file.schema)

Downloaded file: C:\Users\Ghada\.cache\huggingface\hub\datasets--Exgentic--agent-llm-traces\snapshots\70036b93a04e61b0ea2706a68b962f4f26774587\data\train-00000-of-00039.parquet

Number of rows: 46
Number of columns: 35

Dataset schema:
required group field_id=-1 schema {
  optional binary field_id=-1 harness (String);
  optional binary field_id=-1 benchmark (String);
  optional group field_id=-1 models (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional int64 field_id=-1 max_tokens;
  optional int64 field_id=-1 total_tokens;
  optional binary field_id=-1 session_id (String);
  optional group field_id=-1 spans (List) {
    repeated group field_id=-1 list {
      optional group field_id=-1 element {
        optional binary field_id=-1 span_id (String);
        optional binary field_id=-1 name (String);
        optional binary field_id=-1 kind (String);
        optional binary field_id=-1 start_time (String);
        opti

In [10]:
#juste un affichage

import pandas as pd

df_sample = pd.read_parquet(file_path)

print("Dataset shape:", df_sample.shape)

display(df_sample.head(3))

Dataset shape: (46, 8)


,harness,benchmark,models,max_tokens,total_tokens,session_id,spans,collected_at
0,claude_code,appworld,[azure/DeepSeek-V3.2],119938,3144046,2a21e94e0687_1b28ad67,"[{'span_id': '2f8989c28bd888d7', 'name': 'chat...",2026-05-07T09:04:59.116973
1,claude_code,appworld,[azure/DeepSeek-V3.2],118007,1965091,2a21e94e0687_23955e09,"[{'span_id': 'e2ac8974ea706ff1', 'name': 'chat...",2026-05-07T09:04:55.395117
2,claude_code,appworld,[azure/DeepSeek-V3.2],118394,2314684,2a21e94e0687_29595dd0,"[{'span_id': 'f180f76b0e1b4ffd', 'name': 'chat...",2026-05-07T09:04:59.853183


In [11]:
# 1. Display all column names
print("=== ALL DATASET COLUMNS ===")

for i, column in enumerate(df_sample.columns, start=1):
    print(f"{i}. {column}")

# 2. Display the data types
print("\n=== DATA TYPES ===")
print(df_sample.dtypes)

# 3. Inspect the first record without its nested spans
print("\n=== FIRST RECORD (TOP-LEVEL FIELDS) ===")

first_record = df_sample.iloc[0]

for column in df_sample.columns:
    if column != "spans":
        print(f"{column}: {first_record[column]}")

=== ALL DATASET COLUMNS ===
1. harness
2. benchmark
3. models
4. max_tokens
5. total_tokens
6. session_id
7. spans
8. collected_at

=== DATA TYPES ===
harness            str
benchmark          str
models          object
max_tokens       int64
total_tokens     int64
session_id         str
spans           object
collected_at       str
dtype: object

=== FIRST RECORD (TOP-LEVEL FIELDS) ===
harness: claude_code
benchmark: appworld
models: ['azure/DeepSeek-V3.2']
max_tokens: 119938
total_tokens: 3144046
session_id: 2a21e94e0687_1b28ad67
collected_at: 2026-05-07T09:04:59.116973


In [12]:
# Inspect spans in the first record

spans = first_record["spans"]

print("Number of spans:", len(spans))

print("\nFirst 3 spans:")

for span in spans[:3]:
    print(span)
    print("-" * 50)

Number of spans: 28

First 3 spans:
{'span_id': '2f8989c28bd888d7', 'name': 'chat azure/DeepSeek-V3.2', 'kind': 'SPAN_KIND_CLIENT', 'start_time': '2026-04-20T06:42:30.382312+00:00', 'end_time': '2026-04-20T06:42:51.081693+00:00', 'attributes': {'gen_ai.operation.name': 'chat', 'gen_ai.request.model': 'azure/DeepSeek-V3.2', 'gen_ai.response.model': 'azure/DeepSeek-V3.2', 'gen_ai.usage.input_tokens': 112838, 'gen_ai.usage.output_tokens': 244, 'gen_ai.response.id': '5aa6e25481924622ba6aa4e051dd15bf', 'gen_ai.response.finish_reasons': array(['tool_calls'], dtype=object), 'gen_ai.input.messages': '[{"role": "user", "parts": [{"type": "text", "content": "Context: {\'policy\': \\"This environment provides a set of applications, each exposing a predefined set of APIs that may be used to perform tasks on behalf of the supervisor. The applications include: supervisor, amazon, phone, file_system, spotify, venmo, gmail, splitwise, simple_note, todoist.\\\\n The available applications and their API

In [13]:
# Select the first session
first_record = df_sample.iloc[0]

# Extract its spans
spans = first_record["spans"]

print("Session ID:", first_record["session_id"])
print("Number of spans:", len(spans))

# Display the structure of the first span
print("\n=== FIRST SPAN ===")

first_span = spans[0]

for key, value in first_span.items():
    print(f"{key}: {value}")

Session ID: 2a21e94e0687_1b28ad67
Number of spans: 28

=== FIRST SPAN ===
span_id: 2f8989c28bd888d7
name: chat azure/DeepSeek-V3.2
kind: SPAN_KIND_CLIENT
start_time: 2026-04-20T06:42:30.382312+00:00
end_time: 2026-04-20T06:42:51.081693+00:00
attributes: {'gen_ai.operation.name': 'chat', 'gen_ai.request.model': 'azure/DeepSeek-V3.2', 'gen_ai.response.model': 'azure/DeepSeek-V3.2', 'gen_ai.usage.input_tokens': 112838, 'gen_ai.usage.output_tokens': 244, 'gen_ai.response.id': '5aa6e25481924622ba6aa4e051dd15bf', 'gen_ai.response.finish_reasons': array(['tool_calls'], dtype=object), 'gen_ai.input.messages': '[{"role": "user", "parts": [{"type": "text", "content": "Context: {\'policy\': \\"This environment provides a set of applications, each exposing a predefined set of APIs that may be used to perform tasks on behalf of the supervisor. The applications include: supervisor, amazon, phone, file_system, spotify, venmo, gmail, splitwise, simple_note, todoist.\\\\n The available applications and

In [14]:
# Compare the structure of the first 5 spans

for i, span in enumerate(spans[:5], start=1):

    print(f"\n========== SPAN {i} ==========")

    print("Name:", span.get("name"))
    print("Kind:", span.get("kind"))
    print("Start time:", span.get("start_time"))
    print("End time:", span.get("end_time"))
    print("Status:", span.get("status"))

    print("Attributes:")
    print(span.get("attributes"))


========== SPAN 1 ==========
Name: chat azure/DeepSeek-V3.2
Kind: SPAN_KIND_CLIENT
Start time: 2026-04-20T06:42:30.382312+00:00
End time: 2026-04-20T06:42:51.081693+00:00
Status: {'code': 1, 'message': ''}
Attributes:
{'gen_ai.operation.name': 'chat', 'gen_ai.request.model': 'azure/DeepSeek-V3.2', 'gen_ai.response.model': 'azure/DeepSeek-V3.2', 'gen_ai.usage.input_tokens': 112838, 'gen_ai.usage.output_tokens': 244, 'gen_ai.response.id': '5aa6e25481924622ba6aa4e051dd15bf', 'gen_ai.response.finish_reasons': array(['tool_calls'], dtype=object), 'gen_ai.input.messages': '[{"role": "user", "parts": [{"type": "text", "content": "Context: {\'policy\': \\"This environment provides a set of applications, each exposing a predefined set of APIs that may be used to perform tasks on behalf of the supervisor. The applications include: supervisor, amazon, phone, file_system, spotify, venmo, gmail, splitwise, simple_note, todoist.\\\\n The available applications and their APIs are fixed for the task.

In [15]:
import pandas as pd

# Extract spans from the first session
spans = df_sample.iloc[0]["spans"] #cellule de spans pour la première ligne ( ligne =session ) 

span_summary = []

for span in spans:
    attributes = span.get("attributes", {})

    start = pd.to_datetime(span.get("start_time"), utc=True)
    end = pd.to_datetime(span.get("end_time"), utc=True)

    duration = (end - start).total_seconds()

    span_summary.append({
        "span_id": span.get("span_id"),
        "name": span.get("name"),
        "kind": span.get("kind"),
        "operation": attributes.get("gen_ai.operation.name"),
        "model": attributes.get("gen_ai.request.model"),
        "input_tokens": attributes.get("gen_ai.usage.input_tokens"),
        "output_tokens": attributes.get("gen_ai.usage.output_tokens"),
        "duration_seconds": duration,
        "status": span.get("status")
    })

spans_df = pd.DataFrame(span_summary)

display(spans_df.head(10))

,span_id,name,kind,operation,model,input_tokens,output_tokens,duration_seconds,status
0,2f8989c28bd888d7,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,112838,244,20.699381,"{'code': 1, 'message': ''}"
1,4adc2d5e17873fca,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,113129,246,6.038253,"{'code': 1, 'message': ''}"
2,4c638ed5174d3184,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,113422,42,5.395831,"{'code': 1, 'message': ''}"
3,6df9bc0e84b3ba73,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,113728,249,17.744476,"{'code': 1, 'message': ''}"
4,12d12b0d2559bfe2,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114021,59,4.096330,"{'code': 1, 'message': ''}"
5,f7a900c1d56a92a5,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114119,73,50.357742,"{'code': 1, 'message': ''}"
6,1632363737154edc,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114212,41,46.639015,"{'code': 1, 'message': ''}"
7,b140e4b3f2a156a9,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114473,77,27.280142,"{'code': 1, 'message': ''}"
8,45d3d82c55093f59,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114674,163,127.542007,"{'code': 1, 'message': ''}"
9,868efef01d4bc964,chat azure/DeepSeek-V3.2,SPAN_KIND_CLIENT,chat,azure/DeepSeek-V3.2,114854,147,49.931547,"{'code': 1, 'message': ''}"


In [16]:
#recuperer tous les spans
all_spans = []

for _, row in df_sample.iterrows():
    for span in row["spans"]:
        all_spans.append(span)

print("Total spans:", len(all_spans))

Total spans: 925


In [17]:
#afficher tous les champs de spans
span_fields = set()

for span in all_spans:
    span_fields.update(span.keys())

print("=== ALL SPAN FIELDS ===")

for field in sorted(span_fields):
    print(field)

print("Total fields:", len(span_fields))

=== ALL SPAN FIELDS ===
attributes
benchmark
end_time
harness
kind
models
name
resource_attributes
session_id
span_id
start_time
status
trace_id
type
Total fields: 14


In [18]:
# afficher tous les attributs de spans
attribute_keys = set()

for span in all_spans:
    attributes = span.get("attributes") or {}
    attribute_keys.update(attributes.keys())

print("=== ALL ATTRIBUTE KEYS ===")

for key in sorted(attribute_keys):
    print(key)

print("Total attributes:", len(attribute_keys))

=== ALL ATTRIBUTE KEYS ===
gen_ai.input.messages
gen_ai.operation.name
gen_ai.output.messages
gen_ai.request.model
gen_ai.response.finish_reasons
gen_ai.response.id
gen_ai.response.model
gen_ai.tool.definitions
gen_ai.usage.input_tokens
gen_ai.usage.output_tokens
Total attributes: 10


In [19]:
#verifier la presence de chaque champ dans les spans

from collections import Counter

field_presence = Counter()

for span in all_spans:
    field_presence.update(span.keys())

print("=== FIELD PRESENCE ===")

for field, count in field_presence.most_common():
    percentage = count / len(all_spans) * 100
    print(f"{field}: {count}/{len(all_spans)} ({percentage:.1f}%)")

=== FIELD PRESENCE ===
span_id: 925/925 (100.0%)
name: 925/925 (100.0%)
kind: 925/925 (100.0%)
start_time: 925/925 (100.0%)
end_time: 925/925 (100.0%)
attributes: 925/925 (100.0%)
resource_attributes: 925/925 (100.0%)
status: 925/925 (100.0%)
type: 925/925 (100.0%)
harness: 925/925 (100.0%)
benchmark: 925/925 (100.0%)
models: 925/925 (100.0%)
session_id: 925/925 (100.0%)
trace_id: 925/925 (100.0%)


In [20]:
print("=== SPAN KINDS ===")

print(Counter(
    span.get("kind", "MISSING")
    for span in all_spans
))

print("\n=== SPAN NAMES ===")

print(Counter(
    span.get("name", "MISSING")
    for span in all_spans
))

print("\n=== SPAN TYPES ===")

print(Counter(
    span.get("type", "MISSING")
    for span in all_spans
))

=== SPAN KINDS ===
Counter({'SPAN_KIND_CLIENT': 925})

=== SPAN NAMES ===
Counter({'chat azure/DeepSeek-V3.2': 497, 'chat azure/Kimi-K2.5': 318, 'chat aws/claude-opus-4-5': 110})

=== SPAN TYPES ===
Counter({'llm_call': 925})


In [21]:
#Examiner les métadonnées des ressources
resource_keys = set()

for span in all_spans:
    resource_attributes = span.get("resource_attributes") or {}
    resource_keys.update(resource_attributes.keys())

print("=== RESOURCE ATTRIBUTE KEYS ===")

for key in sorted(resource_keys):
    print(key)

=== RESOURCE ATTRIBUTE KEYS ===
service.name
service.version
telemetry.sdk.language
telemetry.sdk.name
telemetry.sdk.version


In [22]:
tool_spans = []

for span in all_spans:
    attributes = span.get("attributes") or {}

    if (
        "tool" in str(span.get("name", "")).lower()
        or "tool" in str(span.get("type", "")).lower()
        or "tool" in str(attributes.get("gen_ai.operation.name", "")).lower()
    ):
        tool_spans.append(span)

print("Number of potential tool-related spans:", len(tool_spans))

for span in tool_spans[:3]:
    print("\nName:", span.get("name"))
    print("Type:", span.get("type"))
    print("Operation:", (span.get("attributes") or {}).get("gen_ai.operation.name"))

Number of potential tool-related spans: 0


In [23]:
# verification de la presence de finish_reasons et tool_definitions_count (appel aux outils)
from collections import Counter

finish_reasons = Counter()
tool_definitions_count = 0

for span in all_spans:
    attributes = span.get("attributes") or {}

    reasons = attributes.get("gen_ai.response.finish_reasons")

    if reasons is not None:
        for reason in reasons:
            finish_reasons[str(reason)] += 1

    if attributes.get("gen_ai.tool.definitions") is not None:
        tool_definitions_count += 1

print("=== FINISH REASONS ===")
print(finish_reasons)

print("\nSpans with tool definitions:", tool_definitions_count)

=== FINISH REASONS ===
Counter({'tool_calls': 757, 'stop': 116})

Spans with tool definitions: 917


In [24]:
#verifier les status 
status_counts = Counter()

for span in all_spans:
    status = span.get("status") or {}
    status_counts[str(status.get("code"))] += 1

print("=== STATUS CODES ===")

for code, count in status_counts.items():
    print(f"Code {code}: {count}")

=== STATUS CODES ===
Code 1: 873
Code 2: 52


In [25]:
print("=== RESOURCE ATTRIBUTES EXAMPLE ===")

print(all_spans[0].get("resource_attributes"))

print("\n=== SPAN TYPE ===")
print(all_spans[0].get("type"))

print("\n=== MODEL VALUES ===")
print(Counter(
    (span.get("attributes") or {}).get("gen_ai.request.model")
    for span in all_spans
))

=== RESOURCE ATTRIBUTES EXAMPLE ===
{'telemetry.sdk.language': 'python', 'telemetry.sdk.name': 'opentelemetry', 'telemetry.sdk.version': '1.40.0', 'service.name': 'litellm', 'service.version': '1.0.0'}

=== SPAN TYPE ===
llm_call

=== MODEL VALUES ===
Counter({'azure/DeepSeek-V3.2': 497, 'azure/Kimi-K2.5': 318, 'aws/claude-opus-4-5': 110})


In [26]:
import json
from collections import Counter

tool_calls = Counter()
tool_call_spans = 0
unreadable_messages = 0

def find_tool_calls(obj):
    """Search recursively for tool_call objects."""
    if isinstance(obj, dict):
        if obj.get("type") == "tool_call":
            name = obj.get("name", "unknown")
            yield name

        for value in obj.values():
            yield from find_tool_calls(value)

    elif isinstance(obj, list):
        for item in obj:
            yield from find_tool_calls(item)


for span in all_spans:
    attributes = span.get("attributes") or {}
    raw_messages = attributes.get("gen_ai.output.messages")

    if raw_messages is None:
        continue

    try:
        if isinstance(raw_messages, str):
            messages = json.loads(raw_messages)
        else:
            messages = raw_messages

        names = list(find_tool_calls(messages))

        if names:
            tool_call_spans += 1
            tool_calls.update(names)

    except (json.JSONDecodeError, TypeError):
        unreadable_messages += 1


print("=== TOOL CALL ANALYSIS ===")
print("Spans containing tool calls:", tool_call_spans)
print("Total tool calls detected:", sum(tool_calls.values()))
print("Tool names:", tool_calls)
print("Unreadable messages:", unreadable_messages)

=== TOOL CALL ANALYSIS ===
Spans containing tool calls: 865
Total tool calls detected: 1160
Tool names: Counter({'mcp__environment__simple_note__search_notes': 117, 'TodoWrite': 107, 'mcp__environment__simple_note__show_note': 85, 'mcp__environment__venmo__show_transactions': 68, 'mcp__environment__venmo__login': 60, 'mcp__environment__supervisor__show_account_passwords': 47, 'mcp__environment__finish': 45, 'mcp__environment__venmo__show_received_payment_requests': 40, 'mcp__environment__file_system__move_file': 40, 'mcp__environment__spotify__show_song': 37, 'mcp__environment__phone__search_contacts': 31, 'mcp__environment__venmo__create_payment_request': 27, 'mcp__environment__spotify__like_song': 22, 'mcp__environment__todoist__show_project': 22, 'mcp__environment__phone__login': 20, 'mcp__environment__venmo__approve_payment_request': 20, 'mcp__environment__file_system__delete_directory': 20, 'mcp__environment__venmo__search_friends': 18, 'mcp__environment__simple_note__login': 16, 

In [27]:
from huggingface_hub import HfApi

DATASET_NAME = "Exgentic/agent-llm-traces"

api = HfApi()

dataset_info = api.dataset_info(
    repo_id=DATASET_NAME,
    files_metadata=True
)

parquet_files = [
    file for file in dataset_info.siblings #recupere tous les fichiers du dataset dans une liste
    if file.rfilename.endswith(".parquet") #conserver uniquement les fichiers Parquet
]

print("Number of Parquet files:", len(parquet_files)) 

total_size = 0

for file in parquet_files:
    size = file.size

    if size is not None:
        total_size += size

    print(
        file.rfilename,
        "| Size:",
        f"{size / (1024 ** 2):.2f} MB" if size is not None else "Unknown"
    )

print("\nTotal size:", round(total_size / (1024 ** 2), 2), "MB")

Number of Parquet files: 39
data/train-00000-of-00039.parquet | Size: 23.03 MB
data/train-00001-of-00039.parquet | Size: 19.80 MB
data/train-00002-of-00039.parquet | Size: 14.61 MB
data/train-00003-of-00039.parquet | Size: 2.50 MB
data/train-00004-of-00039.parquet | Size: 4.06 MB
data/train-00005-of-00039.parquet | Size: 8.56 MB
data/train-00006-of-00039.parquet | Size: 16.72 MB
data/train-00007-of-00039.parquet | Size: 15.25 MB
data/train-00008-of-00039.parquet | Size: 38.49 MB
data/train-00009-of-00039.parquet | Size: 130.15 MB
data/train-00010-of-00039.parquet | Size: 130.26 MB
data/train-00011-of-00039.parquet | Size: 130.81 MB
data/train-00012-of-00039.parquet | Size: 39.13 MB
data/train-00013-of-00039.parquet | Size: 14.18 MB
data/train-00014-of-00039.parquet | Size: 14.67 MB
data/train-00015-of-00039.parquet | Size: 17.32 MB
data/train-00016-of-00039.parquet | Size: 15.41 MB
data/train-00017-of-00039.parquet | Size: 72.12 MB
data/train-00018-of-00039.parquet | Size: 89.85 MB
dat

In [28]:
import pandas as pd

inventory = pd.DataFrame([
    {
        "file_name": file.rfilename,
        "size_bytes": file.size,
        "size_mb": round(file.size / (1024 ** 2), 2)
        if file.size is not None else None
    }
    for file in parquet_files
])

# Trier les fichiers du plus grand au plus petit
inventory = inventory.sort_values(
    by="size_mb",
    ascending=False
)

# Afficher les 10 fichiers les plus volumineux
display(inventory.head(10))

# Enregistrer le rapport
inventory.to_csv(
    "../reports/parquet_inventory.csv",
    index=False
)

print("Inventory saved successfully!")

,file_name,size_bytes,size_mb
11,data/train-00011-of-00039.parquet,137160236,130.81
10,data/train-00010-of-00039.parquet,136589137,130.26
9,data/train-00009-of-00039.parquet,136468470,130.15
18,data/train-00018-of-00039.parquet,94218331,89.85
19,data/train-00019-of-00039.parquet,92017332,87.75
17,data/train-00017-of-00039.parquet,75624930,72.12
12,data/train-00012-of-00039.parquet,41029485,39.13
8,data/train-00008-of-00039.parquet,40355325,38.49
0,data/train-00000-of-00039.parquet,24144062,23.03
1,data/train-00001-of-00039.parquet,20761819,19.80


Inventory saved successfully!


In [29]:
#check the schema of each Parquet file in the dataset
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from collections import Counter

schema_results = []

for file in parquet_files:

    print("Checking:", file.rfilename)

    file_path = hf_hub_download(
        repo_id=DATASET_NAME,
        filename=file.rfilename,
        repo_type="dataset"
    )

    parquet_file = pq.ParquetFile(file_path)

    schema = parquet_file.schema_arrow

    schema_results.append({
        "file_name": file.rfilename,
        "num_rows": parquet_file.metadata.num_rows,
        "num_columns": len(schema.names),
        "columns": list(schema.names)
    })

print("\nSchema inspection completed!")

Checking: data/train-00000-of-00039.parquet
Checking: data/train-00001-of-00039.parquet
Checking: data/train-00002-of-00039.parquet
Checking: data/train-00003-of-00039.parquet
Checking: data/train-00004-of-00039.parquet
Checking: data/train-00005-of-00039.parquet
Checking: data/train-00006-of-00039.parquet
Checking: data/train-00007-of-00039.parquet
Checking: data/train-00008-of-00039.parquet
Checking: data/train-00009-of-00039.parquet
Checking: data/train-00010-of-00039.parquet
Checking: data/train-00011-of-00039.parquet
Checking: data/train-00012-of-00039.parquet
Checking: data/train-00013-of-00039.parquet
Checking: data/train-00014-of-00039.parquet
Checking: data/train-00015-of-00039.parquet
Checking: data/train-00016-of-00039.parquet
Checking: data/train-00017-of-00039.parquet
Checking: data/train-00018-of-00039.parquet
Checking: data/train-00019-of-00039.parquet
Checking: data/train-00020-of-00039.parquet
Checking: data/train-00021-of-00039.parquet
Checking: data/train-00022-of-00

In [30]:
schema_df = pd.DataFrame(schema_results)

display(
    schema_df[
        ["file_name", "num_rows", "num_columns"]
    ]
)

print("\nTotal rows:", schema_df["num_rows"].sum())

print(
    "\nDifferent column counts:",
    schema_df["num_columns"].unique()
)

,file_name,num_rows,num_columns
0,data/train-00000-of-00039.parquet,46,8
1,data/train-00001-of-00039.parquet,46,8
2,data/train-00002-of-00039.parquet,46,8
3,data/train-00003-of-00039.parquet,46,8
4,data/train-00004-of-00039.parquet,46,8
5,data/train-00005-of-00039.parquet,46,8
6,data/train-00006-of-00039.parquet,46,8
7,data/train-00007-of-00039.parquet,46,8
8,data/train-00008-of-00039.parquet,46,8
9,data/train-00009-of-00039.parquet,46,8



Total rows: 1781

Different column counts: [8]


In [31]:
reference_columns = set(schema_results[0]["columns"])

for result in schema_results:

    current_columns = set(result["columns"])

    missing = reference_columns - current_columns
    extra = current_columns - reference_columns

    if missing or extra:
        print("\nSchema difference:", result["file_name"])
        print("Missing columns:", missing)
        print("Extra columns:", extra)

print("\nReference schema:", sorted(reference_columns))


Reference schema: ['benchmark', 'collected_at', 'harness', 'max_tokens', 'models', 'session_id', 'spans', 'total_tokens']


Examiner la structure imbriquée de spans

In [32]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

nested_schema_results = [] # Store the nested schema information

for file in parquet_files:

    file_path = hf_hub_download(
        repo_id=DATASET_NAME,
        filename=file.rfilename,
        repo_type="dataset"
    )

    parquet_file = pq.ParquetFile(file_path)

    # Récupérer le type de la colonne spans
    spans_type = parquet_file.schema_arrow.field("spans").type

    nested_schema_results.append({
        "file_name": file.rfilename,
        "spans_schema": str(spans_type)
    })

print("Nested schema inspection completed!")

Nested schema inspection completed!


In [33]:
nested_schemas = [
    result["spans_schema"]
    for result in nested_schema_results
]

unique_schemas = set(nested_schemas)

print("Number of different spans schemas:", len(unique_schemas))

if len(unique_schemas) == 1:
    print("All files have the same nested spans schema.")
else:
    print("Schema differences detected!")

    for index, schema in enumerate(unique_schemas, start=1):
        print(f"\nSchema {index}:\n{schema}")

Number of different spans schemas: 1
All files have the same nested spans schema.


In [34]:
# Afficher le schéma imbriqué du premier fichier
print(nested_schema_results[0]["spans_schema"])

list<element: struct<span_id: string, name: string, kind: string, start_time: string, end_time: string, attributes: struct<gen_ai.operation.name: string, gen_ai.request.model: string, gen_ai.response.model: string, gen_ai.usage.input_tokens: int64, gen_ai.usage.output_tokens: int64, gen_ai.response.id: string, gen_ai.response.finish_reasons: list<element: string>, gen_ai.input.messages: string, gen_ai.output.messages: string, gen_ai.tool.definitions: string>, resource_attributes: struct<telemetry.sdk.language: string, telemetry.sdk.name: string, telemetry.sdk.version: string, service.name: string, service.version: string>, status: struct<code: int64, message: string>, type: string, harness: string, benchmark: string, models: list<element: string>, session_id: string, trace_id: string>>


In [35]:
#que pour le premier fichier 
# Vérifier les colonnes principales
print("Columns:")
print(df_sample.columns.tolist())

# Vérifier les valeurs manquantes
print("\nMissing values:")
print(df_sample.isnull().sum())

# Vérifier les types de données
print("\nData types:")
print(df_sample.dtypes)

Columns:
['harness', 'benchmark', 'models', 'max_tokens', 'total_tokens', 'session_id', 'spans', 'collected_at']

Missing values:
harness         0
benchmark       0
models          0
max_tokens      0
total_tokens    0
session_id      0
spans           0
collected_at    0
dtype: int64

Data types:
harness            str
benchmark          str
models          object
max_tokens       int64
total_tokens     int64
session_id         str
spans           object
collected_at       str
dtype: object


In [36]:
# Vérifier la cohérence entre le niveau session et le niveau span

mismatches = []

for _, row in df_sample.iterrows():
    for span in row["spans"]:
        if row["harness"] != span["harness"]:
            mismatches.append({
                "session_id": row["session_id"],
                "field": "harness",
                "session_value": row["harness"],
                "span_value": span["harness"]
            })

        if row["benchmark"] != span["benchmark"]:
            mismatches.append({
                "session_id": row["session_id"],
                "field": "benchmark",
                "session_value": row["benchmark"],
                "span_value": span["benchmark"]
            })

print("Number of mismatches:", len(mismatches))
print(mismatches[:10])

Number of mismatches: 0
[]


In [ ]:

# 1. Nombre total de lignes
print("Total rows:", len(df_sample))

# 2. Vérifier les doublons de session_id
print(
    "Duplicate session_id:",
    df_sample["session_id"].duplicated().sum()
)

# 3. Nombre de sessions uniques
print(
    "Unique session_id:",
    df_sample["session_id"].nunique()
)

# 4. Vérifier la diversité des données
print("\nNumber of harness:", df_sample["harness"].nunique())

print("Number of benchmarks:", df_sample["benchmark"].nunique())

print(
    "Number of models:",
    df_sample["models"].explode().nunique()
)

Total rows: 46
Duplicate session_id: 0
Unique session_id: 46

Number of harness: 1
Number of benchmarks: 1
Number of models: 3


In [38]:
# Valeurs distinctes de harness et benchmark
print("Harness values:")
print(df_sample["harness"].value_counts())

print("\nBenchmark values:")
print(df_sample["benchmark"].value_counts())

# Répartition des modèles
models_exploded = df_sample.explode("models")

print("\nModel distribution:")
print(models_exploded["models"].value_counts())

Harness values:
harness
claude_code    46
Name: count, dtype: int64

Benchmark values:
benchmark
appworld    46
Name: count, dtype: int64

Model distribution:
models
azure/DeepSeek-V3.2    25
azure/Kimi-K2.5        18
aws/claude-opus-4-5     3
Name: count, dtype: int64


In [39]:
from collections import Counter
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

# Compteurs
attribute_counts = Counter()
total_spans = 0

# Parcourir les 39 fichiers
for i, file in enumerate(parquet_files, start=1):

    print(f"Processing file {i}/39: {file.rfilename}")

    file_path = hf_hub_download(
        repo_id=DATASET_NAME,
        filename=file.rfilename,
        repo_type="dataset"
    )

    parquet_file = pq.ParquetFile(file_path)

    # Lire uniquement la colonne spans, par petits lots
    for batch in parquet_file.iter_batches(
        columns=["spans"],
        batch_size=10
    ):
        sessions_spans = batch.column(0).to_pylist()

        for spans in sessions_spans:
            for span in spans or []:

                total_spans += 1

                attributes = span.get("attributes") or {}

                for name, value in attributes.items():
                    if value is not None:
                        attribute_counts[name] += 1


# Afficher les résultats
print("\n===== ATTRIBUTE ANALYSIS =====")
print("Total spans:", total_spans)

for name, count in sorted(attribute_counts.items()):
    percentage = (count / total_spans) * 100

    print(
        f"{name}: {count} spans "
        f"({percentage:.2f}%)"
    )

Processing file 1/39: data/train-00000-of-00039.parquet
Processing file 2/39: data/train-00001-of-00039.parquet
Processing file 3/39: data/train-00002-of-00039.parquet
Processing file 4/39: data/train-00003-of-00039.parquet
Processing file 5/39: data/train-00004-of-00039.parquet
Processing file 6/39: data/train-00005-of-00039.parquet
Processing file 7/39: data/train-00006-of-00039.parquet
Processing file 8/39: data/train-00007-of-00039.parquet
Processing file 9/39: data/train-00008-of-00039.parquet
Processing file 10/39: data/train-00009-of-00039.parquet
Processing file 11/39: data/train-00010-of-00039.parquet
Processing file 12/39: data/train-00011-of-00039.parquet
Processing file 13/39: data/train-00012-of-00039.parquet
Processing file 14/39: data/train-00013-of-00039.parquet
Processing file 15/39: data/train-00014-of-00039.parquet
Processing file 16/39: data/train-00015-of-00039.parquet
Processing file 17/39: data/train-00016-of-00039.parquet
Processing file 18/39: data/train-00017-

In [40]:
import pandas as pd

attributes_df = pd.DataFrame([
    {
        "attribute": name,
        "spans_with_value": count,
        "percentage": round(count / total_spans * 100, 2)
    }
    for name, count in attribute_counts.items()
])

attributes_df = attributes_df.sort_values(
    by="percentage",
    ascending=False
).reset_index(drop=True)

display(attributes_df)

,attribute,spans_with_value,percentage
0,gen_ai.operation.name,38052,100.00
1,gen_ai.request.model,38052,100.00
2,gen_ai.response.model,38052,100.00
3,gen_ai.usage.input_tokens,38052,100.00
4,gen_ai.usage.output_tokens,38052,100.00
5,gen_ai.input.messages,38052,100.00
6,gen_ai.response.id,36713,96.48
7,gen_ai.response.finish_reasons,36713,96.48
8,gen_ai.output.messages,36713,96.48
9,gen_ai.tool.definitions,28008,73.60


In [41]:
#verifiant les valeurs nulles et vides pour chaque attribut dans les spans
from collections import Counter

value_types = Counter()
empty_counts = Counter()

for file in parquet_files:

    file_path = hf_hub_download(
        repo_id=DATASET_NAME,
        filename=file.rfilename,
        repo_type="dataset"
    )

    parquet_file = pq.ParquetFile(file_path)

    for batch in parquet_file.iter_batches(
        columns=["spans"],
        batch_size=10
    ):
        for spans in batch.column(0).to_pylist():

            for span in spans or []:
                attributes = span.get("attributes") or {}

                for name, value in attributes.items():

                    if value is None:
                        value_types[(name, "None")] += 1

                    elif isinstance(value, str) and value.strip() == "":
                        empty_counts[name] += 1

                    elif isinstance(value, (list, dict)) and len(value) == 0:
                        empty_counts[name] += 1

                    else:
                        value_types[(name, "Non-empty")] += 1

print("===== EMPTY VALUES =====")

for name, count in sorted(empty_counts.items()):
    print(f"{name}: {count} empty values")

print("\n===== NULL VALUES =====")

for (name, value_type), count in sorted(value_types.items()):
    if value_type == "None":
        print(f"{name}: {count} None values")

===== EMPTY VALUES =====

===== NULL VALUES =====
gen_ai.output.messages: 1339 None values
gen_ai.response.finish_reasons: 1339 None values
gen_ai.response.id: 1339 None values
gen_ai.tool.definitions: 10044 None values


In [43]:
import json
from collections import Counter

tool_counter = Counter()
spans_with_tool_calls = 0
total_tool_calls = 0

L’objectif est de répondre à trois questions :
1. Combien de spans contiennent au moins un Tool Call ?
2. Combien de Tool Calls ont été demandés au total ?
3. Quels outils sont les plus fréquemment demandés ?

In [44]:
import json
from collections import Counter

tool_counter = Counter()
spans_with_tool_calls = 0
total_tool_calls = 0

In [ ]:
#fct pour detecter les appels aux outils 
def find_tool_calls(obj):
    found = []

    if isinstance(obj, dict):
        if obj.get("type") == "tool_call":
            found.append(obj)

        for value in obj.values():
            found.extend(find_tool_calls(value))

    elif isinstance(obj, list):
        for item in obj:
            found.extend(find_tool_calls(item))

    return found

In [46]:
import pyarrow.parquet as pq
from pathlib import Path

parquet_files = sorted(Path("../data/raw").glob("*.parquet"))

for file in parquet_files:
    table = pq.read_table(
        file,
        columns=["spans"]
    )

    for row in table.to_pylist():
        for span in row["spans"] or []:

            attributes = span.get("attributes") or {}
            output_messages = attributes.get("gen_ai.output.messages")

            if not output_messages:
                continue

            try:
                messages = json.loads(output_messages)
            except (json.JSONDecodeError, TypeError):
                continue

            tool_calls = find_tool_calls(messages)

            if tool_calls:
                spans_with_tool_calls += 1
                total_tool_calls += len(tool_calls)

                for call in tool_calls:
                    tool_name = call.get("name", "unknown")
                    tool_counter[tool_name] += 1

In [53]:
from pathlib import Path
import pyarrow.parquet as pq
import json
from collections import Counter

parquet_files = sorted(
    (Path.home() / ".cache" / "huggingface").rglob("*.parquet")
)

tool_counter_global = Counter()
spans_with_tool_calls_global = 0
total_tool_calls_global = 0
total_spans_global = 0

for file in parquet_files:

    table = pq.read_table(file, columns=["spans"])

    for row in table.to_pylist():

        for span in row["spans"] or []:

            total_spans_global += 1

            attributes = span.get("attributes") or {}
            output = attributes.get("gen_ai.output.messages")

            if output is None:
                continue

            try:
                messages = json.loads(output)
            except (json.JSONDecodeError, TypeError):
                continue

            span_tool_calls = []

            for message in messages:
                for part in message.get("parts", []) or []:

                    if part.get("type") == "tool_call":
                        span_tool_calls.append(part)

            if span_tool_calls:
                spans_with_tool_calls_global += 1
                total_tool_calls_global += len(span_tool_calls)

                for call in span_tool_calls:
                    tool_name = call.get("name", "unknown")
                    tool_counter_global[tool_name] += 1

print("===== GLOBAL TOOL CALL ANALYSIS =====")

print("Files analyzed:", len(parquet_files))
print("Total spans:", total_spans_global)
print("Spans containing tool calls:", spans_with_tool_calls_global)
print("Total tool calls:", total_tool_calls_global)

print("\n===== TOOL FREQUENCY =====")

for tool, count in tool_counter_global.most_common():
    print(f"{tool}: {count}")

===== GLOBAL TOOL CALL ANALYSIS =====
Files analyzed: 39
Total spans: 38052
Spans containing tool calls: 25565
Total tool calls: 33390

===== TOOL FREQUENCY =====
mcp__environment__bash: 8551
mcp__environment__search: 3186
mcp__environment__message: 1685
bash: 1548
spotify__show_song: 1088
TodoWrite: 1034
simple_note__search_notes: 883
mcp__environment__simple_note__show_note: 862
simple_note__show_note: 837
venmo__show_transactions: 760
mcp__environment__simple_note__search_notes: 552
mcp__environment__get_order_details: 424
finish: 400
todoist__show_project: 391
mcp__environment__finish: 328
mcp__environment__venmo__show_transactions: 322
supervisor__show_account_passwords: 303
file_system__delete_file: 243
venmo__approve_payment_request: 234
mcp__environment__get_user_details: 226
phone__search_contacts: 222
mcp__environment__get_reservation_details: 186
spotify__add_to_queue: 177
venmo__login: 166
mcp__environment__get_product_details: 164
mcp__environment__get_document: 153
can_se

In [54]:
import pandas as pd

tool_df = pd.DataFrame(
    tool_counter_global.most_common(),
    columns=["tool_name", "call_count"]
)

tool_df["percentage"] = (
    tool_df["call_count"] / total_tool_calls_global * 100
).round(2)

print(tool_df.head(20))

tool_df.to_csv(
    "../reports/tool_call_frequency.csv",
    index=False
)

print("\nReport saved successfully!")

                                      tool_name  call_count  percentage
0                        mcp__environment__bash        8551       25.61
1                      mcp__environment__search        3186        9.54
2                     mcp__environment__message        1685        5.05
3                                          bash        1548        4.64
4                            spotify__show_song        1088        3.26
5                                     TodoWrite        1034        3.10
6                     simple_note__search_notes         883        2.64
7      mcp__environment__simple_note__show_note         862        2.58
8                        simple_note__show_note         837        2.51
9                      venmo__show_transactions         760        2.28
10  mcp__environment__simple_note__search_notes         552        1.65
11          mcp__environment__get_order_details         424        1.27
12                                       finish         400     

analyse des distributions catégorielles globales

In [ ]:
#analyze the distribution of harness, benchmark, and models across all Parquet files
from pathlib import Path
import pyarrow.parquet as pq
from collections import Counter

# Locate the 39 Parquet files
cache_path = Path.home() / ".cache" / "huggingface"

parquet_files = sorted(cache_path.rglob("*.parquet"))

# Initialize counters
harness_counter = Counter()
benchmark_counter = Counter()
model_counter = Counter()

total_sessions = 0

# Analyze all files
for file in parquet_files:

    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["harness", "benchmark", "models"],
        batch_size=100
    ):

        for row in batch.to_pylist():

            total_sessions += 1

            # Count harness
            if row["harness"] is not None:
                harness_counter[row["harness"]] += 1

            # Count benchmark
            if row["benchmark"] is not None:
                benchmark_counter[row["benchmark"]] += 1

            # Count models once per session
            models = row["models"] or []

            for model in set(models):
                model_counter[model] += 1


print("Total sessions:", total_sessions)

print("\n===== HARNESS DISTRIBUTION =====")
for name, count in harness_counter.most_common():
    print(f"{name}: {count}")

print("\n===== BENCHMARK DISTRIBUTION =====")
for name, count in benchmark_counter.most_common():
    print(f"{name}: {count}")

print("\n===== MODEL DISTRIBUTION =====")
for name, count in model_counter.most_common():
    print(f"{name}: {count}")

Total sessions: 1781

===== HARNESS DISTRIBUTION =====
claude_code: 839
tool_calling: 431
smolagents_code: 315
openai_solo: 114
tool_calling_with_shortlisting: 82

===== BENCHMARK DISTRIBUTION =====
tau2_retail: 469
appworld: 406
swebench: 391
tau2_airline: 196
tau2_telecom: 186
browsecompplus: 133

===== MODEL DISTRIBUTION =====
openai/Azure/gpt-4.1: 550
aws/claude-opus-4-5: 509
Azure/gpt-4.1: 365
gcp/gemini-3-pro-preview: 361
azure/Kimi-K2.5: 158
azure/DeepSeek-V3.2: 128
Azure/gpt-5.2-2025-12-11: 37
openai/azure/Kimi-K2.5: 29
openai/azure/DeepSeek-V3.2: 18
gpt-5.2-2025-12-11: 15


Analyser les catégories des Spans :cette etape nous permettra de savoir quelles opérations sont représentées dans les traces, comment les spans sont catégorisés et quelles raisons de terminaison sont enregistrées.


In [56]:

from collections import Counter
from pathlib import Path
import pyarrow.parquet as pq

kind_counter = Counter()
type_counter = Counter()
status_counter = Counter()
finish_reason_counter = Counter()

total_spans = 0

cache_path = Path.home() / ".cache" / "huggingface"
parquet_files = sorted(cache_path.rglob("*.parquet"))

In [57]:
for file in parquet_files:

    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["spans"],
        batch_size=100
    ):

        for row in batch.to_pylist():

            for span in row["spans"] or []:

                total_spans += 1

                # Span kind
                kind = span.get("kind")
                if kind is not None:
                    kind_counter[kind] += 1

                # Span type
                span_type = span.get("type")
                if span_type is not None:
                    type_counter[span_type] += 1

                # Status code
                status = span.get("status") or {}
                status_code = status.get("code")

                if status_code is not None:
                    status_counter[status_code] += 1

                # Finish reasons
                attributes = span.get("attributes") or {}
                finish_reasons = attributes.get(
                    "gen_ai.response.finish_reasons"
                )

                if finish_reasons is not None:
                    for reason in finish_reasons:
                        finish_reason_counter[reason] += 1

In [58]:
print("Total spans:", total_spans)

print("\n===== KIND DISTRIBUTION =====")
for name, count in kind_counter.most_common():
    print(f"{name}: {count}")

print("\n===== TYPE DISTRIBUTION =====")
for name, count in type_counter.most_common():
    print(f"{name}: {count}")

print("\n===== STATUS CODE DISTRIBUTION =====")
for code, count in status_counter.most_common():
    print(f"{code}: {count}")

print("\n===== FINISH REASONS DISTRIBUTION =====")
for reason, count in finish_reason_counter.most_common():
    print(f"{reason}: {count}")

Total spans: 38052

===== KIND DISTRIBUTION =====
SPAN_KIND_CLIENT: 38052

===== TYPE DISTRIBUTION =====
llm_call: 38052

===== STATUS CODE DISTRIBUTION =====
1: 36713
2: 1339

===== FINISH REASONS DISTRIBUTION =====
stop: 25317
tool_calls: 11394
content_filter: 2


Vérification des doublons et identifiants. session_id , span_id , trace_id

In [59]:
#verifier les session_id 
from pathlib import Path
import pyarrow.parquet as pq
from collections import Counter

cache_path = Path.home() / ".cache" / "huggingface"
parquet_files = sorted(cache_path.rglob("*.parquet"))

session_counter = Counter()
total_rows = 0
missing_session_ids = 0

for file in parquet_files:

    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["session_id"],
        batch_size=100
    ):

        for row in batch.to_pylist():

            total_rows += 1

            session_id = row["session_id"]

            if session_id is None:
                missing_session_ids += 1
            else:
                session_counter[session_id] += 1


# Calculate duplicate statistics
unique_sessions = len(session_counter)

duplicate_ids = {
    session_id: count
    for session_id, count in session_counter.items()
    if count > 1
}

duplicate_occurrences = sum(
    count - 1 for count in duplicate_ids.values()
)

print("===== SESSION ID ANALYSIS =====")

print("Total sessions:", total_rows)
print("Missing session IDs:", missing_session_ids)
print("Unique session IDs:", unique_sessions)
print("Duplicated session IDs:", len(duplicate_ids))
print("Duplicate occurrences:", duplicate_occurrences)

print("\n===== DUPLICATE EXAMPLES =====")

for session_id, count in list(duplicate_ids.items())[:10]:
    print(session_id, ":", count)

===== SESSION ID ANALYSIS =====
Total sessions: 1781
Missing session IDs: 0
Unique session IDs: 1781
Duplicated session IDs: 0
Duplicate occurrences: 0

===== DUPLICATE EXAMPLES =====


In [60]:
#verifiant span_id et trace_id 
from collections import Counter
from pathlib import Path
import pyarrow.parquet as pq

span_id_counter = Counter()
trace_id_counter = Counter()

total_spans = 0
missing_span_ids = 0
missing_trace_ids = 0

for file in parquet_files:

    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["spans"],
        batch_size=100
    ):

        for row in batch.to_pylist():

            for span in row["spans"] or []:

                total_spans += 1

                span_id = span.get("span_id")
                trace_id = span.get("trace_id")

                if span_id is None:
                    missing_span_ids += 1
                else:
                    span_id_counter[span_id] += 1

                if trace_id is None:
                    missing_trace_ids += 1
                else:
                    trace_id_counter[trace_id] += 1


# Analyze duplicates
duplicate_span_ids = {
    key: count
    for key, count in span_id_counter.items()
    if count > 1
}

duplicate_trace_ids = {
    key: count
    for key, count in trace_id_counter.items()
    if count > 1
}

print("===== SPAN ID ANALYSIS =====")
print("Total spans:", total_spans)
print("Missing span IDs:", missing_span_ids)
print("Unique span IDs:", len(span_id_counter))
print("Duplicated span IDs:", len(duplicate_span_ids))
print("Duplicate occurrences:", sum(
    count - 1 for count in duplicate_span_ids.values()
))

print("\n===== TRACE ID ANALYSIS =====")
print("Total spans:", total_spans)
print("Missing trace IDs:", missing_trace_ids)
print("Unique trace IDs:", len(trace_id_counter))
print("Trace IDs appearing multiple times:", len(duplicate_trace_ids))
print("Repeated occurrences:", sum(
    count - 1 for count in duplicate_trace_ids.values()
))

===== SPAN ID ANALYSIS =====
Total spans: 38052
Missing span IDs: 0
Unique span IDs: 33524
Duplicated span IDs: 4528
Duplicate occurrences: 4528

===== TRACE ID ANALYSIS =====
Total spans: 38052
Missing trace IDs: 0
Unique trace IDs: 1781
Trace IDs appearing multiple times: 1781
Repeated occurrences: 36271


In [61]:
#Comprendre les doublons de span_id
from collections import defaultdict

span_trace_mapping = defaultdict(set)
span_trace_counts = Counter()

for file in parquet_files:

    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["spans"],
        batch_size=100
    ):

        for row in batch.to_pylist():

            for span in row["spans"] or []:

                span_id = span.get("span_id")
                trace_id = span.get("trace_id")

                if span_id is not None and trace_id is not None:
                    span_trace_mapping[span_id].add(trace_id)
                    span_trace_counts[(span_id, trace_id)] += 1


# Same span_id in multiple traces
span_ids_multiple_traces = {
    span_id: traces
    for span_id, traces in span_trace_mapping.items()
    if len(traces) > 1
}

# Same span_id repeated inside the same trace
span_ids_repeated_same_trace = {
    key: count
    for key, count in span_trace_counts.items()
    if count > 1
}

print("===== SPAN ID COLLISION ANALYSIS =====")

print(
    "Span IDs appearing in multiple traces:",
    len(span_ids_multiple_traces)
)

print(
    "Span IDs repeated within the same trace:",
    len(span_ids_repeated_same_trace)
)

print("\n===== EXAMPLES =====")

for span_id, traces in list(span_ids_multiple_traces.items())[:5]:
    print("Span ID:", span_id)
    print("Number of different traces:", len(traces))

===== SPAN ID COLLISION ANALYSIS =====
Span IDs appearing in multiple traces: 4528
Span IDs repeated within the same trace: 0

===== EXAMPLES =====
Span ID: 1372e223ddf02bd4
Number of different traces: 2
Span ID: fd08c435b2a5ebd7
Number of different traces: 2
Span ID: c803a2e3e2719f60
Number of different traces: 2
Span ID: 861509cab41778f8
Number of different traces: 2
Span ID: 0697d069b63cd06f
Number of different traces: 2


vérifier la cohérence Session–Trace–Span

In [62]:
from collections import defaultdict

session_to_traces = defaultdict(set)
trace_to_sessions = defaultdict(set)

for file in parquet_files:
    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["session_id", "spans"],
        batch_size=100
    ):
        for row in batch.to_pylist():
            session_id = row["session_id"]

            for span in row["spans"] or []:
                trace_id = span.get("trace_id")

                if session_id is not None and trace_id is not None:
                    session_to_traces[session_id].add(trace_id)
                    trace_to_sessions[trace_id].add(session_id)

sessions_multiple_traces = {
    session: traces
    for session, traces in session_to_traces.items()
    if len(traces) > 1
}

traces_multiple_sessions = {
    trace: sessions
    for trace, sessions in trace_to_sessions.items()
    if len(sessions) > 1
}

print("===== SESSION-TRACE CONSISTENCY =====")
print("Sessions analyzed:", len(session_to_traces))
print("Sessions linked to multiple traces:", len(sessions_multiple_traces))
print("Traces linked to multiple sessions:", len(traces_multiple_sessions))

===== SESSION-TRACE CONSISTENCY =====
Sessions analyzed: 1781
Sessions linked to multiple traces: 0
Traces linked to multiple sessions: 0


session_id consistency check between session level and span level

In [63]:

total_spans = 0
matching_session_ids = 0
mismatching_session_ids = 0
missing_span_session_ids = 0

for file in parquet_files:
    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["session_id", "spans"],
        batch_size=100
    ):
        for row in batch.to_pylist():
            row_session_id = row["session_id"]

            for span in row["spans"] or []:
                total_spans += 1

                span_session_id = span.get("session_id")

                if span_session_id is None:
                    missing_span_session_ids += 1

                elif span_session_id == row_session_id:
                    matching_session_ids += 1

                else:
                    mismatching_session_ids += 1

print("===== SESSION ID CONSISTENCY =====")
print("Total spans:", total_spans)
print("Matching session IDs:", matching_session_ids)
print("Mismatching session IDs:", mismatching_session_ids)
print("Missing span session IDs:", missing_span_session_ids)

===== SESSION ID CONSISTENCY =====
Total spans: 38052
Matching session IDs: 38052
Mismatching session IDs: 0
Missing span session IDs: 0
